# Baseline ensemble


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from scipy.stats import rankdata

In [4]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed_data_dir = project_root / "data" / "processed"
raw_data_dir = project_root / "data" / "raw"
oof_dir = project_root / "artifacts" / "oof"
test_predictions_dir = project_root / "artifacts" / "test_predictions"
submissions_dir = project_root / "submissions"

train_path = processed_data_dir / "train_with_folds.parquet"
sample_submission_path = raw_data_dir / "sample_submission.csv"

train_df = pd.read_parquet(train_path)
sample_submission_df = pd.read_csv(sample_submission_path)
target = train_df["satisfaction"]

train_df.shape, sample_submission_df.shape, target.shape

((699635, 25), (299844, 2), (699635,))

In [7]:
xgboost_oof = np.load(
    oof_dir / "xgboost_oof.npy"
)

xgboost_oof.shape, np.isnan(xgboost_oof).sum()

((699635,), np.int64(0))

In [8]:
xgboost_auc = roc_auc_score(
    target,
    xgboost_oof
)

xgboost_auc

0.9588211425501422

In [10]:
lightgbm_oof = np.load(
    oof_dir / "lightgbm.npy"
)

lightgbm_oof.shape, np.isnan(lightgbm_oof).sum()

((699635,), np.int64(0))

In [11]:
lightgbm_auc = roc_auc_score(
    target,
    lightgbm_oof
)

lightgbm_auc

0.9588781741592428

In [12]:
catboost_oof = np.load(
    oof_dir / "catboost.npy"
)

catboost_oof.shape, np.isnan(catboost_oof).sum()

((699635,), np.int64(0))

In [13]:
catboost_auc = roc_auc_score(
    target,
    catboost_oof
)

catboost_auc

0.9587125217844891

In [14]:
ensemble_oof = (xgboost_oof + lightgbm_oof + catboost_oof) / 3

ensemble_auc = roc_auc_score(
    target,
    ensemble_oof
)

ensemble_auc

0.9591024784845751

In [15]:

xgboost_rank = rankdata(xgboost_oof) / len(xgboost_oof)
lightgbm_rank = rankdata(lightgbm_oof) / len(lightgbm_oof)
catboost_rank = rankdata(catboost_oof) / len(catboost_oof)

rank_oof = (
    xgboost_rank
    + lightgbm_rank
    + catboost_rank
) / 3

rank_auc = roc_auc_score(
    target,
    rank_oof
)

rank_auc

0.9591271310879411

In [18]:
rank_coef_oof = (0.3 * xgboost_rank
    + 0.4 * lightgbm_rank
    + 0.3 * catboost_rank
)

rank_coef_auc = roc_auc_score(
    target,
    rank_coef_oof
)

rank_coef_auc

0.9591291107801281

In [19]:
xgboost_test = np.load(test_predictions_dir / "xgboost_test.npy")
lightgbm_test = np.load(test_predictions_dir / "lightgbm.npy")
catboost_test = np.load(test_predictions_dir / "catboost.npy")

print("xgboost:", xgboost_test.shape, np.isnan(xgboost_test).sum())
print("lightgbm:", lightgbm_test.shape, np.isnan(lightgbm_test).sum())
print("catboost:", catboost_test.shape, np.isnan(catboost_test).sum())

xgboost: (299844,) 0
lightgbm: (299844,) 0
catboost: (299844,) 0


In [23]:
xgboost_test_rank = rankdata(xgboost_test) / len(xgboost_test)
lightgbm_test_rank = rankdata(lightgbm_test) / len(lightgbm_test)
catboost_test_rank = rankdata(catboost_test) / len(catboost_test)

weighted_rank_test = (0.3 * xgboost_test_rank
    + 0.4 * lightgbm_test_rank
    + 0.3 * catboost_test_rank
)


weighted_rank_test.shape, np.isnan(weighted_rank_test).sum(), weighted_rank_test.min(), weighted_rank_test.max()

((299844,),
 np.int64(0),
 np.float64(9.671695948559918e-06),
 np.float64(0.9999829911554008))

In [24]:
submission = sample_submission_df.copy()
submission["satisfaction"] = weighted_rank_test

submissions_dir.mkdir(parents=True, exist_ok=True)
submission_path = submissions_dir / "ensemble_rank_30_40_30.csv"
submission.to_csv(submission_path, index=False)

print("saved:", submission_path)
print("shape:", submission.shape)
print("columns:", submission.columns.tolist())
print("missing:", submission["satisfaction"].isna().sum())
submission.head()

saved: c:\Users\User\Desktop\Airline Satisfaction\submissions\ensemble_rank_30_40_30.csv
shape: (299844, 2)
columns: ['id', 'satisfaction']
missing: 0


,id,satisfaction
0,699635,0.517996
1,699636,0.620520
2,699637,0.627193
3,699638,0.432728
4,699639,0.386658


In [ ]:
weighted_probability_oof = (0.3 * xgboost_oof
    + 0.4 * lightgbm_oof
    + 0.3 * catboost_oof
    )

weighted_probability_auc = roc_auc_score(
    target,
    weighted_probability_oof
)

weighted_probability_test = (0.3 * xgboost_test
    + 0.4 * lightgbm_test
    + 0.3 * catboost_test
    )
weighted_probability_test.shape

(299844,)

In [1]:
from pathlib import Path
from io import BytesIO
import zipfile

import numpy as np
import pandas as pd
from scipy.stats import rankdata


project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent


xgboost_test = np.load(
    project_dir / "artifacts/test_predictions/xgboost_test.npy"
)

lightgbm_test = np.load(
    project_dir / "artifacts/test_predictions/lightgbm.npy"
)

catboost_test = np.load(
    project_dir / "artifacts/test_predictions/catboost.npy"
)


archive_path = project_dir / "artifacts/realmlp_final.zip"

with zipfile.ZipFile(archive_path) as archive:
    realmlp_test = np.load(
        BytesIO(archive.read("realmlp_test.npy"))
    )

    realmlp_test_ids = np.load(
        BytesIO(archive.read("test_ids.npy"))
    )


xgboost_rank = rankdata(xgboost_test) / len(xgboost_test)
lightgbm_rank = rankdata(lightgbm_test) / len(lightgbm_test)
catboost_rank = rankdata(catboost_test) / len(catboost_test)
realmlp_rank = rankdata(realmlp_test) / len(realmlp_test)


tree_rank_prediction = (
    0.30 * xgboost_rank
    + 0.40 * lightgbm_rank
    + 0.30 * catboost_rank
)


final_prediction = (
    0.78 * tree_rank_prediction
    + 0.22 * realmlp_rank
)


submission = pd.read_csv(
    project_dir / "data/raw/sample_submission.csv"
)

assert len(submission) == len(final_prediction)
assert np.array_equal(
    submission["id"].to_numpy(),
    realmlp_test_ids,
)
assert not np.isnan(final_prediction).any()

submission["satisfaction"] = final_prediction

submission_path = (
    project_dir
    / "submissions/ensemble_trees_realmlp_22.csv"
)

submission.to_csv(submission_path, index=False)

print("Файл сохранён:", submission_path)
print("Размер:", submission.shape)
submission.head()

Файл сохранён: c:\Users\User\Desktop\Airline Satisfaction\submissions\ensemble_trees_realmlp_22.csv
Размер: (299844, 2)


,id,satisfaction
0,699635,0.521139
1,699636,0.614595
2,699637,0.635093
3,699638,0.440351
4,699639,0.392685
